In [78]:
import pandas as pd
import numpy as np
import seaborn as sns
import csv
import matplotlib.pyplot as plt
import optuna
import scipy.stats as stats
import phik
import warnings
import json

from pandas.api.types import is_string_dtype
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.utils import resample
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from optbinning import BinningProcess
from tqdm import tqdm
from catboost import CatBoostClassifier

TARGET = 'target'
ID_COL = 'reco_id_curr'
RANDOM_STATE = 42

def split_xy(frame):
    y = frame[TARGET].astype(int)
    X = frame.drop(columns=[TARGET, ID_COL], errors='ignore')
    return X, y

def gini(y_true, y_pred):
    auc = roc_auc_score(y_true, y_pred)
    return 2 * auc  - 1


In [23]:
df = pd.read_csv('Data/train.csv')

In [24]:
data = df.copy()

# Разбиение данных

In [25]:
# RANDOM_STATE = 42
# Сначала отделяем 10% для late_test
train_val_test, data_late = train_test_split(
    data,
    test_size=0.1,  # 10% на late_test
    stratify=data['target'],
    random_state=RANDOM_STATE
)

# Из оставшихся 90% берем 70% для train (это ~77.8% от 90%)
# train_size = 0.7 / 0.9 ≈ 0.7778
data_train, val_test = train_test_split(
    train_val_test,
    train_size=0.7778,  # 70% от ВСЕХ данных
    stratify=train_val_test['target'],
    random_state=RANDOM_STATE
)

# Из оставшихся 20% делим поровну на val и test
data_val, data_test = train_test_split(
    val_test,
    test_size=0.5,  # 10% от ВСЕХ данных
    stratify=val_test['target'],
    random_state=RANDOM_STATE
)

In [26]:
# Бьём данные на X и y

X_train, y_train = split_xy(data_train)
X_val, y_val = split_xy(data_val)
X_test, y_test = split_xy(data_test)
X_late, y_late = split_xy(data_late)

# Собираем топы по IV, F-Score и FSTR

In [27]:
# IV, F-Score
ord_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_train_for_f = X_train.fillna(X_train.median(numeric_only=True))
for column in X_train_for_f.columns:
    X_train_for_f[column] = X_train[column].fillna(X_train[column].mode()[0])

def feature_scoring(col: str) -> dict:
    result = {'feature': col, 'IV': 0.0, 'F_score': 0.0, 'p_value': np.nan}
    X_one = X_train_for_f[[col]]
    y_one = y_train
    if X_one[col].nunique() < 2:
        return result

    if is_string_dtype(X_train_for_f[col]):
        X_one[col] = ord_enc.fit_transform(X_one[[col]])

    f_sel = SelectKBest(score_func=f_classif, k='all')
    f_sel.fit(X_one, y_one)
    result['F_score'] = float(f_sel.scores_[0])
    result['p_value'] = float(f_sel.pvalues_[0])

    bp = BinningProcess(variable_names=[col])
    bp.fit(X_one, y_one)
    result['IV'] = float(bp.get_binned_variable(col).binning_table.iv)
    return result

rows = [feature_scoring(col) for col in tqdm(X_train_for_f.columns)]
scores = pd.DataFrame(rows)
iv_table = scores[['feature', 'IV']].sort_values('IV', ascending=False).reset_index(drop=True)
f_table = scores[['feature', 'F_score', 'p_value']].sort_values('F_score', ascending=False).reset_index(drop=True)

100%|██████████| 120/120 [00:10<00:00, 11.81it/s]


In [30]:
# FSTR

# Спец обработка для catboost (заменяем пропуски на 'missing')
cat_cols = X_train.select_dtypes(include=['object', 'str']).columns.tolist()

X_train[cat_cols] = X_train[cat_cols].fillna('missing').astype(str)
X_val[cat_cols] = X_val[cat_cols].fillna('missing').astype(str)
X_test[cat_cols] = X_test[cat_cols].fillna('missing').astype(str)
X_late[cat_cols] = X_late[cat_cols].fillna('missing').astype(str)

best_parameters_for_cb = {
    'iterations' : 50,
    'learning_rate' : 0.1,
    'depth' : 6,
    'random_state' : RANDOM_STATE
}

model_cb = CatBoostClassifier(**best_parameters_for_cb, cat_features=(X_train.select_dtypes(include=['str']).columns).tolist())

model_cb.fit(X_train, y_train, verbose=False)

fstr_table = pd.DataFrame({'importance' : model_cb.get_feature_importance(), 'feature' : X_train.columns.tolist()})

In [58]:
# Перебор топ сколько брать из каждой колнки (берём топ n из IV, F-Score и FSTR и объединяем)
def top_n_f(n):
    combined_set = set(f_table['feature'].tolist()[:n]) | set(iv_table['feature'].tolist()[:n]) | set(fstr_table['feature'].tolist()[:n])
    return list(combined_set)

top_n = []
top = []
for n in tqdm(range(24, 25)):
    cols = top_n_f(n)
    cat_features = X_train[cols].select_dtypes(include=['object', 'category', 'str']).columns.tolist()
    model_cb = CatBoostClassifier(**best_parameters_for_cb, cat_features=cat_features)
    model_cb.fit(X_train[cols], y_train, verbose=False)
    probabilities = model_cb.predict_proba(X_val[cols])[:, 1]
    gini1 = gini(y_val, probabilities)
    probabilities = model_cb.predict_proba(X_test[cols])[:, 1]
    gini2 = gini(y_test, probabilities)
    top.append([n, len(cols), gini1, gini2])
with open("top.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(top)

100%|██████████| 1/1 [00:05<00:00,  5.70s/it]


In [40]:
featurs_stage_2 = top_n_f(24)
X_train_2 = X_train[featurs_stage_2]
X_val_2 = X_val[featurs_stage_2]
X_test_2 = X_test[featurs_stage_2]
X_late_2 = X_late[featurs_stage_2]

In [41]:
model_cb = CatBoostClassifier(**best_parameters_for_cb, cat_features=(X_train_67.select_dtypes(include=['str']).columns).tolist())

model_cb.fit(X_train_2, y_train, verbose=False)

probabilities = model_cb.predict_proba(X_val_2)[:,1]
gini(y_val, probabilities)

0.5117318086569975

In [65]:
# корреляции
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

corrs = []
# Спирмен Пиросн Крамер

warnings.filterwarnings('ignore')

for col1 in X_train_2.columns:
    corrs_row = []
    for col2 in X_train_2.columns:
        cor = [0, 0, 0]
        if (is_string_dtype(X_train_2[col1]) == is_string_dtype(X_train_2[col2]) and is_string_dtype(X_train_2[col2])):
            confusion_matrix = pd.crosstab(X_train_2[col1], X_train_2[col2])
            chi2 = stats.chi2_contingency(confusion_matrix)[0]
            n = confusion_matrix.to_numpy().sum()
            min_dim = min(confusion_matrix.shape) - 1
            cramers_v = np.sqrt(chi2 / (n * min_dim))
            cor[2] = cramers_v
        if (is_string_dtype(X_train_2[col1]) == is_string_dtype(X_train_2[col2]) and not is_string_dtype(X_train_2[col2])):
            p_corr = X_train_2[col1].corr(X_train_2[col2], method="pearson")
            s_corr = X_train_2[col1].corr(X_train_2[col2], method="spearman")
            cor[0] = p_corr
            cor[1] = s_corr
        corrs_row.append(cor)
    corrs.append(corrs_row)
np.save("corrs.npy", corrs)

In [70]:
corr_target = []
data_2 = X_train_2.copy()
data_2['target'] = y_train
target_corr = data_2.phik_matrix(interval_cols=num_cols)['target']
target_corr = target_corr.drop(index='target').sort_values(ascending=False)
corr_with_target = target_corr.reset_index().rename(columns={'index': 'feature', 'target': 'phik_correlation'})

In [66]:
phik_matrix = X_train_2.phik_matrix(interval_cols=X_train_2.select_dtypes(include=['int', 'float']).columns.tolist())

In [89]:
# Берём самый коррелирующий с target, удаляем корелирующие с ним более чем на max_cor, пока размер меньше sz
def get_good_cols(sz, max_cor):
    ans = []
    available_features = corr_with_target['feature'].tolist()
    while available_features and len(ans) < sz:
        best_col = available_features[0]
        ans.append(best_col)
        high_corr_mask = phik_matrix[best_col].abs() > max_cor
        correlated_cols = phik_matrix.index[high_corr_mask].tolist()
        available_features = [col for col in available_features if col not in correlated_cols]
    return ans

In [90]:
# Должен быть перебор, но по ощущения 24, 0.7 норм
featurs_stage_3 = get_good_cols(24, 0.7)
model_cb = CatBoostClassifier(**best_parameters_for_cb, cat_features=(X_train[featurs_stage_3].select_dtypes(include=['str']).columns).tolist())

model_cb.fit(X_train[featurs_stage_3], y_train, verbose=False)

probabilities = model_cb.predict_proba(X_val[featurs_stage_3])[:,1]
print(gini(y_val, probabilities))
probabilities = model_cb.predict_proba(X_test[featurs_stage_3])[:,1]
print(gini(y_test, probabilities))
probabilities = model_cb.predict_proba(X_late[featurs_stage_3])[:,1]
print(gini(y_late, probabilities))

0.5053523020736237
0.48745112440607685
0.48954717049278607


In [91]:
X_train[featurs_stage_3].to_csv('X_train_cb.csv', index=False)
X_val[featurs_stage_3].to_csv('X_val_cb.csv', index=False)
X_test[featurs_stage_3].to_csv('X_test_cb.csv', index=False)
X_late[featurs_stage_3].to_csv('X_late_cb.csv', index=False)
y_train.to_csv('y_train_cb.csv', index=False)
y_val.to_csv('y_val_cb.csv', index=False)
y_test.to_csv('y_test_cb.csv', index=False)
y_late.to_csv('y_late_cb.csv', index=False)

In [92]:
# Для графиков
probabilities = model_cb.predict_proba(X_val[featurs_stage_3])[:,1]
auc_score = roc_auc_score(y_val, probabilities)
gini_score = 2 * auc_score - 1
fpr, tpr, thresholds = roc_curve(y_val, probabilities)

np.save("gini_score.npy", gini_score)
np.save("auc_score.npy", auc_score)
np.save("fpr.npy", fpr)
np.save("tpr.npy", tpr)
np.save("thresholds.npy", thresholds)